# Steefel & MacQuarrie (1996) Figure 6 replication

This notebook visualizes the first-order decay benchmark generated by `run.py`. The solid black curve is the analytical semi-infinite advection–dispersion–decay solution. The model contains an explicit $x=0$ CNC boundary node; the reported numerical profiles are the following interior nodes, $x=j\Delta x$. A separate panel retains the raw MODFLOW DIS cell-center coordinates as a grid-coordinate sensitivity.

Generate or refresh the data first from the repository root:

```text
python examples/ex019_Splitting_KineticDecay1D/run.py
```

In [ ]:
import sys
from pathlib import Path

sys.dont_write_bytecode = True
CASE_NAME = "ex019_Splitting_KineticDecay1D"
candidates = (
    candidate
    for base in (Path.cwd(), *Path.cwd().parents)
    for candidate in (base, base / "examples" / CASE_NAME)
)
CASE_DIR = next(
    (
        path.resolve()
        for path in candidates
        if path.name == CASE_NAME and (path / "modflow_model.py").is_file()
    ),
    None,
)
if CASE_DIR is None:
    raise FileNotFoundError(f"Cannot locate examples/{CASE_NAME} from {Path.cwd()}")
EXAMPLES_DIR = CASE_DIR.parent
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))

from example_utils import load_results, read_headings, restore_archives, runtime_path, time_indices

CASE_FILE = CASE_DIR / "run.py"
INPUT_DIR = CASE_DIR / "input_data"
OUTPUT_DIR = runtime_path(CASE_FILE, "output")
SIMULATION_DIR = runtime_path(CASE_FILE, "simulation")
restore_archives(OUTPUT_DIR)
restore_archives(SIMULATION_DIR)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 240,
        "font.size": 10.5,
        "axes.titleweight": "bold",
    }
)


CASE_DIR


## Load and audit the simulation products

The archive contains only simulation output and the analytical reference; this notebook does not rerun MF6PQC.

In [ ]:
with np.load(OUTPUT_DIR / "paper_figure6_data.npz") as archive:
    arrays = {name: archive[name].copy() for name in archive.files}
metrics = pd.read_csv(OUTPUT_DIR / "paper_figure6_metrics.csv")

required = {
    "x_paper_nodes_m",
    "x_modflow_cell_centers_m",
    "x_analytical_dense_m",
    "analytical_dense",
    "analytical_paper_nodes",
    "analytical_cell_centers",
    "profile__SNIA__cfl_1",
    "profile__Strang__cfl_1",
    "profile__SIA__cfl_1",
}
missing = sorted(required - arrays.keys())
if missing:
    raise RuntimeError(f"Incomplete benchmark archive; missing: {missing}")
if metrics.empty or not np.isfinite(metrics["paper_node_rmse"]).all():
    raise RuntimeError("Metrics are empty or contain non-finite errors.")

display(
    metrics.sort_values(["method", "cfl"])[
        [
            "method",
            "cfl",
            "logical_steps",
            "damkohler_per_step",
            "paper_node_rmse",
            "cell_center_rmse",
            "boundary_endpoint_concentration",
            "transport_solves",
            "reaction_evaluations",
            "wall_time_seconds",
        ]
    ].reset_index(drop=True)
)


## Paper-style concentration profiles

The original Figure 6 shows SNIA and Strang at CFL = 0.1, 0.5, and 1, and states that SIA nearly overlays the analytical curve. The third panel makes that SIA comparison explicit.

MF6PQC now uses the paper's instantaneous endpoint-rate form for SIA and the MODFLOW CENTRAL stencil for this Pe=2 verification. SIA nearly overlays the analytical curve at every tested CFL; at CFL=1 the accuracy order is SIA, Strang, SNIA. SNIA approaches the reference only after its time step is reduced. Concentrations are normalized by the actual PhreeqcRM inlet value, so the comparison is C/C0.

In [ ]:
def cfl_token(cfl):
    return format(float(cfl), "g").replace(".", "p")


def profile_key(method, cfl):
    return f"profile__{method}__cfl_{cfl_token(cfl)}"


x_nodes = arrays["x_paper_nodes_m"]
x_dense = arrays["x_analytical_dense_m"]
analytical_dense = arrays["analytical_dense"]
styles = {
    0.1: dict(color="#4c78a8", linestyle=":", marker=None),
    0.5: dict(color="#f58518", linestyle="--", marker="o"),
    1.0: dict(color="#54a24b", linestyle="-.", marker="s"),
}

figure, axes = plt.subplots(1, 3, figsize=(14.2, 4.25), sharey=True, constrained_layout=True)
for axis, method in zip(axes[:2], ("SNIA", "Strang"), strict=False):
    axis.plot(x_dense, analytical_dense, color="black", lw=2.0, label="Analytical")
    for cfl in (0.1, 0.5, 1.0):
        key = profile_key(method, cfl)
        axis.plot(x_nodes, arrays[key], lw=1.6, ms=4.0, label=f"CFL = {cfl:g}", **styles[cfl])
    axis.set_title(method)

axis = axes[2]
axis.plot(x_dense, analytical_dense, color="black", lw=2.0, label="Analytical")
sia_rows = metrics.loc[metrics["method"].eq("SIA")].sort_values("cfl")
for cfl in sia_rows["cfl"]:
    key = profile_key("SIA", cfl)
    axis.plot(x_nodes, arrays[key], lw=1.7, ms=4.2, label=f"CFL = {cfl:g}", **styles[float(cfl)])
axis.set_title("SIA (explicit comparison)")

for axis in axes:
    axis.set(xlabel="Distance, x (m)", xlim=(0, 6), ylim=(0, 1.02))
    axis.legend(frameon=True, fontsize=9)
axes[0].set_ylabel(r"Normalized concentration, $C/C_0$")
figure.suptitle("First-order decay splitting benchmark at t = 0.5 yr", fontsize=13)
figure.savefig(OUTPUT_DIR / "steefel1996_figure6_replication.png", bbox_inches="tight")
plt.show()


## Error, coordinate sensitivity, and computational work

The left panel is the temporal/CFL study. The center panel prevents an accuracy claim from hiding its computational cost. The right panel compares the explicit conceptual node coordinates with the raw MODFLOW DIS cell centers; the latter is a grid-coordinate diagnostic, not the Figure-6 validation norm.

In [ ]:
method_colors = {"SNIA": "#4c78a8", "Strang": "#f58518", "SIA": "#54a24b"}
figure, axes = plt.subplots(1, 3, figsize=(14.5, 4.3), constrained_layout=True)

for method, group in metrics.groupby("method", sort=False):
    group = group.sort_values("cfl")
    axes[0].loglog(
        group["cfl"], group["paper_node_rmse"], "o-", color=method_colors[method], label=method
    )
axes[0].set(
    xlabel="CFL number",
    ylabel="RMSE (paper-node convention)",
    title="Error under coupling refinement",
)
axes[0].legend()

for method, group in metrics.groupby("method", sort=False):
    axes[1].loglog(
        group["transport_solves"],
        group["paper_node_rmse"],
        "o",
        ms=7,
        color=method_colors[method],
        label=method,
    )
    for row in group.itertuples():
        axes[1].annotate(
            f"{row.cfl:g}",
            (row.transport_solves, row.paper_node_rmse),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )
axes[1].set(xlabel="Transport solves", ylabel="RMSE", title="Accuracy–work trade-off")
axes[1].legend()

cfl_one = (
    metrics.loc[np.isclose(metrics["cfl"], 1.0)].set_index("method").loc[["SNIA", "Strang", "SIA"]]
)
positions = np.arange(3)
width = 0.36
axes[2].bar(positions - width / 2, cfl_one["paper_node_rmse"], width, label="Paper nodes")
axes[2].bar(positions + width / 2, cfl_one["cell_center_rmse"], width, label="Raw DIS centers")
axes[2].set_xticks(positions, cfl_one.index)
axes[2].set(ylabel="RMSE at CFL = 1", title="Grid-coordinate sensitivity")
axes[2].legend(fontsize=8.5)

for axis in axes:
    axis.grid(True, which="both", alpha=0.25)
figure.savefig(OUTPUT_DIR / "splitting_error_cost.png", bbox_inches="tight")
plt.show()


## Reproduction verdict

In [ ]:
paper_rank = cfl_one["paper_node_rmse"].sort_values()
center_rank = cfl_one["cell_center_rmse"].sort_values()
print("CFL=1 ranking with the paper-node convention:", " < ".join(paper_rank.index))
print("CFL=1 ranking with raw DIS centers:        ", " < ".join(center_rank.index))
print(
    f"SIA paper-node RMSE improvement over Strang: "
    f"{(1 - paper_rank['SIA'] / paper_rank['Strang']) * 100:.2f}%"
)
print(
    "Interpretation: the paper's inlet over-reaction signature is reproduced, "
    "but the method ranking must always be reported together with the spatial "
    "boundary convention and work count."
)


## 论文图 3

以下代码保持论文图的配色、字体、面板尺寸和标注，读取本案例的 `input_data/`、`output/` 与 `simulation/`。结果图保存到 `output/figures/`。

In [ ]:
BASE = OUTPUT_DIR / "figures"
BASE.mkdir(exist_ok=True)

import matplotlib as mpl
from matplotlib.ticker import LogLocator

%matplotlib inline
%config InlineBackend.figure_format = 'svg'
W = 183 / 25.4
COL = ["#477B92", "#C99659", "#986D91"]
TIME4 = ["#477B92", "#6B9B88", "#C99659", "#986D91"]

mpl.rcdefaults()
mpl.rcParams.update(
    {
        "font.family": "Arial",
        "font.size": 9,
        "axes.titlesize": 9,
        "axes.labelsize": 8.5,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,
        "legend.fontsize": 8.5,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "svg.fonttype": "none",
        "axes.linewidth": 0.65,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "legend.frameon": False,
        "xtick.major.size": 2.5,
        "ytick.major.size": 2.5,
        "savefig.facecolor": "white",
    }
)


with np.load(OUTPUT_DIR / "paper_figure6_data.npz", allow_pickle=False) as archive:
    D = {k: archive[k] for k in archive.files}
table = pd.read_csv(OUTPUT_DIR / "paper_figure6_metrics.csv", dtype=str).fillna("")
D["metrics"] = table.to_numpy(dtype=str)
D["metric_columns"] = np.array(table.columns)


def _manuscript_layout_03(fig):
    fig.canvas.draw()
    width_pt, height_pt = fig.get_size_inches() * 72
    offsets = [(0, 0), (0, 0), (0, 0)]
    if len(fig.axes) != len(offsets):
        raise ValueError("Manuscript panel count changed")
    for ax, (dx, dy) in zip(fig.axes, offsets, strict=True):
        pos = ax.get_position()
        ax.set_position([pos.x0 + dx / width_pt, pos.y0 - dy / height_pt, pos.width, pos.height])
    for legend in fig.legends:
        box = legend.get_bbox_to_anchor().transformed(fig.transFigure.inverted())
        legend.set_bbox_to_anchor(
            (box.x0 + 0 / width_pt, box.y0 - 4.5 / height_pt, box.width, box.height),
            transform=fig.transFigure,
        )


def save(n, fig, axes):
    _manuscript_layout_03(fig)
    fig.savefig(BASE / f"figure_{n:02}.png", dpi=600, bbox_inches="tight", pad_inches=0.02)
    plt.show()


def title(ax, letter, name):
    ax.set_title(f"{letter}  {name}", loc="left", pad=7, fontsize=9)


def grid(rows, cols, height, top=0.86, bottom=0.10, wspace=0.50, hspace=0.67):
    fig, axs = plt.subplots(rows, cols, figsize=(W, height), squeeze=False)
    fig.subplots_adjust(left=0.08, right=0.98, top=top, bottom=bottom, wspace=wspace, hspace=hspace)
    return fig, list(axs.flat)


def fig3():
    d = D
    m = d["metrics"]
    names = list(d["metric_columns"])
    fig, axs = grid(1, 3, 2.8, top=0.77, bottom=0.25, wspace=0.48)
    for method, c, marker in zip(["SNIA", "Strang", "SIA"], COL, ["o", "s", "^"], strict=False):
        axs[0].plot(
            d["x_paper_nodes_m"],
            d[f"profile__{method}__cfl_1"],
            color=c,
            marker=marker,
            ms=3,
            mfc="white",
            lw=1.4,
            label=method,
        )
        records = m[m[:, names.index("method")] == method]
        cfl = records[:, names.index("cfl")].astype(float)
        order = np.argsort(cfl)
        error = records[:, names.index("paper_node_rmse")].astype(float)[order]
        solves = records[:, names.index("transport_solves")].astype(float)[order]
        assert np.all(error > 0) and np.all(solves > 0)
        axs[1].plot(cfl[order], error, color=c, marker=marker, ms=4, mfc="white", lw=1.4)
        axs[2].plot(solves, error, color=c, marker=marker, ms=4, mfc="white", lw=1.4)
    axs[0].plot(
        d["x_analytical_dense_m"],
        d["analytical_dense"],
        color=".3",
        ls="--",
        lw=1,
        label="Analytical",
    )
    axs[0].set(
        xlim=(0, 6), ylim=(0, 1.03), xlabel="Distance (m)", ylabel="Normalized concentration"
    )
    axs[1].set(
        xlim=(0.04, 1.06),
        xticks=[0.1, 0.5, 1],
        xlabel="Courant number",
        ylabel="RMSE",
        yscale="log",
    )
    axs[2].set(
        xlim=(100, 10000), xlabel="Transport solves", ylabel="RMSE", xscale="log", yscale="log"
    )
    axs[2].xaxis.set_major_locator(LogLocator(base=10, numticks=3))
    for a in axs[1:]:
        a.set_ylim(0.0025, 0.09)
        a.set_yticks([0.003, 0.01, 0.03])
        a.set_yticklabels(["0.003", "0.01", "0.03"])
        a.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
    for a, letter, t in zip(
        axs,
        "abc",
        ["Concentration at 0.5 yr", "Time step dependence", "Accuracy and cost"],
        strict=False,
    ):
        title(a, letter, t)
    fig.legend(
        *axs[0].get_legend_handles_labels(), ncol=4, loc="upper center", bbox_to_anchor=(0.53, 1)
    )
    save(3, fig, axs)


np.savez_compressed(BASE / "figure_03_data.npz", **D)
fig3()
